# Solución docente · Desafío de evolución de esquema

Este notebook es autocontenido: prepara una Bronze mínima y resuelve el desafío. Usar primero con escala `test`.

In [ ]:
dbutils.widgets.text("student_id", "docente01", "Identificador")
dbutils.widgets.dropdown("scale", "test", ["test", "small", "demo"], "Escala")

In [ ]:
%run ../../common/config

In [ ]:
%run ../../common/generate_data

In [ ]:
from pyspark.sql import functions as F

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
data = build_course_data(spark, config)
paths = write_landing_files(data, config)

## Preparar la tabla histórica

In [ ]:
source_table = f"{config.catalog}.{config.schema}.bronze_events"
historical = (spark.read.json(paths["events_json"])
    .withColumn("_source", F.lit("events_json"))
    .withColumn("_source_file", F.lit(paths["events_json"]))
    .withColumn("_ingested_at", F.current_timestamp()))
historical.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(source_table)

## Diagnosticar el lote nuevo

In [ ]:
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
challenge = spark.read.json(challenge_path)
challenge.printSchema()
display(challenge.groupBy("event_type").count().orderBy("event_type"))
display(challenge.agg(F.count("*").alias("rows"), F.countDistinct("event_id").alias("distinct_ids")))

## Integrar sin modificar la tabla original

La escritura es `overwrite` sobre una tabla derivada; por eso repetir el notebook mantiene el resultado. En la clase 2 esta estrategia se reemplazará por `MERGE` incremental.

In [ ]:
target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"
new_events = (challenge
    .withColumn("_source", F.lit("events_json_v2"))
    .withColumn("_source_file", F.lit(challenge_path))
    .withColumn("_ingested_at", F.current_timestamp()))
combined = (spark.table(source_table)
    .unionByName(new_events, allowMissingColumns=True)
    .dropDuplicates(["event_id"]))
combined.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(target_table)

## Verificación ejecutable

In [ ]:
result = spark.table(target_table)
total_rows = result.count()
distinct_ids = result.select("event_id").distinct().count()
expected_rows = config.rows["events"] + max(100, config.rows["events"] // 20)

assert total_rows == distinct_ids
assert total_rows == expected_rows, (total_rows, expected_rows)
assert "app_version" in result.columns
assert result.where("event_type = 'refund'").count() > 0
assert result.where("app_version IS NULL").count() == config.rows["events"]
print(f"OK: {total_rows:,} eventos únicos en {target_table}")
display(result.groupBy("_source", "event_type").count().orderBy("_source", "event_type"))